# 01 — Dataset EDA

Exploratory analysis of the three HuggingFace datasets used in MedReason-Bench v1.0:
- **MedMCQA** (`openlifescienceai/medmcqa`) — multiple-choice, native subspecialty tags
- **MedQA-USMLE** (`bigbio/med_qa`, config `med_qa_en_4options_bigbio_qa`) — multiple-choice, no native tags
- **PubMedQA** (`qiaojin/PubMedQA`, config `pqa_labeled`) — yes/no/maybe research-question QA

**What this notebook reports** (post-filter, default cap = 100 items):
1. Count per dataset and per `metadata['specialty']` (cardiology / autoimmune / `None`).
2. Stem + question length distributions (chars and approx tokens).
3. A handful of sample items from each dataset, side-by-side, so we can sanity-check the filter.

**To run:** the cells call HF `load_dataset` and require network access + the `datasets` package (`uv pip install -e ".[dev]"`). First run downloads + caches each dataset in `~/.cache/huggingface/`.

In [ ]:
from __future__ import annotations

import sys
from collections import Counter
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from data.loaders.medmcqa import load_medmcqa
from data.loaders.medqa import load_medqa
from data.loaders.pubmedqa import load_pubmedqa

PER_DATASET_LIMIT = 100

## Load each dataset (filtered to subspecialty)

In [ ]:
medmcqa_items = list(load_medmcqa(limit=PER_DATASET_LIMIT))
medqa_items = list(load_medqa(limit=PER_DATASET_LIMIT))
pubmedqa_items = list(load_pubmedqa(limit=PER_DATASET_LIMIT))

all_items = {
    "medmcqa": medmcqa_items,
    "medqa": medqa_items,
    "pubmedqa": pubmedqa_items,
}

for name, items in all_items.items():
    print(f"{name:>10s}: {len(items)} items")

## Counts per subspecialty

In [ ]:
rows = []
for name, items in all_items.items():
    counts = Counter(it.metadata.get("specialty") for it in items)
    rows.append({
        "dataset": name,
        "cardiology": counts.get("cardiology", 0),
        "autoimmune": counts.get("autoimmune", 0),
        "unlabeled": counts.get(None, 0),
        "total": sum(counts.values()),
    })

specialty_df = pd.DataFrame(rows).set_index("dataset")
specialty_df

## Length distributions (stem + question, characters)

In [ ]:
length_rows = []
for name, items in all_items.items():
    for it in items:
        length_rows.append({
            "dataset": name,
            "stem_chars": len(it.stem),
            "question_chars": len(it.question),
            "total_chars": len(it.stem) + len(it.question),
        })

length_df = pd.DataFrame(length_rows)
length_df.groupby("dataset").describe(percentiles=[0.25, 0.5, 0.75, 0.95])

## Sample items (one per dataset)

In [ ]:
for name, items in all_items.items():
    if not items:
        continue
    it = items[0]
    print(f"===== {name} =====")
    print(f"id: {it.id}")
    print(f"specialty: {it.metadata.get('specialty')}")
    if it.stem:
        print(f"stem: {it.stem[:200]}")
    print(f"question: {it.question[:200]}")
    for letter, text in sorted(it.options.items()):
        print(f"  {letter}. {text[:120]}")
    print(f"correct: {it.correct}")
    print()